# RiskAwareRouter: full run on a free Colab T4 (target: under 3 hours)

**Runtime → Change runtime type → T4 GPU** before you start.

| Stage | What runs | Expected time |
|---|---|---|
| Setup | install vLLM, clone repo | ~10-15 min |
| 0 Dataset | download GSM8K / TruthfulQA / MedMCQA, compile 250 C programs | ~3 min |
| 1 Small | Qwen2.5-3B answers ~1,100 queries (batched) | ~10 min |
| 2 Large | Qwen2.5-14B-AWQ answers the same queries | ~30-45 min |
| 3 Grade | script grading + Llama-3.1-8B judge on open answers + severity pre-labels | ~15 min |
| 4 Verify | gcc, Python check, NLI on FAQ answers | ~5-10 min |
| 5 Evaluate | 100 splits, baselines, ablations (CPU) | ~5-15 min |

Every stage writes to Google Drive and skips itself if its output already exists, so a disconnect costs one stage, not the run. Times are estimates: the timing cell below measures yours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/RiskAwareRouter/outputs'
os.makedirs(OUT, exist_ok=True)
os.environ['RAR_OUT'] = OUT

The repo is cloned with a GitHub token if it is private. Add `GITHUB_TOKEN` (repo read access) and `HF_TOKEN` in Colab's **Secrets** panel (key icon on the left). `HF_TOKEN` is needed because Llama 3.1 is gated: accept Meta's licence on its Hugging Face page first. No Llama access? Set `JUDGE_MODEL` to `microsoft/Phi-3.5-mini-instruct` in the next cell.

In [ ]:
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@github.com/Zhussain13/RiskAwareRouting.git' if token else 'https://github.com/Zhussain13/RiskAwareRouting.git'
if not os.path.exists('/content/RiskAwareRouter'):
    !git clone -q {url} /content/RiskAwareRouter
%cd /content/RiskAwareRouter

hf = secret('HF_TOKEN')
if hf:
    os.environ['HF_TOKEN'] = hf
# os.environ['JUDGE_MODEL'] = 'microsoft/Phi-3.5-mini-instruct'   # ungated fallback

In [ ]:
%%time
!pip install -q -r requirements-colab.txt

## Optional: 5-minute timing check
Answers 50 queries with the small model in a scratch folder and extrapolates. Skip it once you trust the estimates.

In [ ]:
%%time
import time, subprocess
env = dict(os.environ, RAR_OUT='/content/timing_check')
subprocess.run(['python', 'build_dataset.py', '--small'], env=env, check=True)
t0 = time.time()
subprocess.run(['python', 'generate.py', '--role', 'small', '--limit', '50'], env=env, check=True)
print(f'50 queries (incl. model load) in {time.time() - t0:.0f}s')

## Stage 0: dataset

In [ ]:
%%time
!python build_dataset.py

## Stage 1: small model (Qwen2.5-3B)

In [ ]:
%%time
!python generate.py --role small

## Stage 2: large model (Qwen2.5-14B-AWQ)

In [ ]:
%%time
!python generate.py --role large

## Stage 3: grading and severity pre-labels (judge model)

In [ ]:
%%time
!python grade.py

## Stage 4: verifiers

In [ ]:
%%time
!python verify.py

## Stage 5: evaluation (CPU)
Uses human severity labels from `data/severity_human.csv` if present; otherwise it falls back to the LLM pre-labels and **says so in every output**. Those results are for development only, not for the paper.

In [ ]:
%%time
!python evaluate.py

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{OUT}/results/summary.md').read()))
display(Image(f'{OUT}/results/cost_vs_risk.png'))

## After this run
1. `python labels.py export` → give `annotator_A.csv` / `annotator_B.csv` to two annotators (rubric: `data/rubric/severity_rubric.md`).
2. `python labels.py merge ...` then `python labels.py finalise` → `data/severity_human.csv`.
3. Hand-check `judge_audit.csv` (10% of judge-graded answers) to report the judge's accuracy.
4. Re-run **only Stage 5**: no GPU needed, every answer is cached in Drive.